# CREATING TABLES

In [0]:
%sql

-- =============================================================================
-- SETUP: Create all Delta tables — Bronze, Silver, Platform
-- PROJECT : azure3_team7_project — Retail Supply Chain & Inventory
-- Run once after 00_setup_catalog_schemas_volumes.sql
-- and 01_setup_audit_tables.sql
-- NOTE: Gold tables will be added after silver is complete and
--       aggregation logic is finalized
-- =============================================================================

USE CATALOG azure3_team7_project;

## BRONZE TABLES
Rules:
- No constraints (nullability not enforced — accept what source sends)
- No ZORDER yet (done in maintenance job after data lands)
- Partitioned by ingested_date for all high-frequency tables
- CDF enabled on all tables for downstream silver streaming reads
- autoOptimize enabled on all tables

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- bronze.customers
-- Source : customers.csv
-- Frequency: Daily
-- preferred_categories arrives as raw JSON array string e.g. ["Books","Clothing"]
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.bronze.customers (
  customer_id           STRING,
  age_group             STRING,
  gender                STRING,
  zip_code              STRING,
  loyalty_tier          STRING,
  first_purchase_date   STRING,
  total_spend           DOUBLE,
  preferred_categories  STRING,   -- raw JSON array string, parsed in silver
  -- audit
  source_file           STRING    NOT NULL,
  ingested_at           TIMESTAMP NOT NULL,
  pipeline_run_id       STRING    NOT NULL
)
USING DELTA
PARTITIONED BY (ingested_date STRING)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Bronze: Customer data (anonymized) from CSV. Append-only.';

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- bronze.product_master
-- Source : product_master.parquet
-- Frequency: Weekly + incremental (SCD Type 2 handled in silver)
-- Not partitioned — small table (50k rows), full reload weekly
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.bronze.product_master (
  product_id      STRING,
  product_name    STRING,
  category        STRING,
  subcategory     STRING,
  brand           STRING,
  supplier_id     STRING,
  cost_price      DOUBLE,
  selling_price   DOUBLE,
  weight          DOUBLE,
  dimensions      STRING,
  status          STRING,
  effective_date  STRING,
  expiry_date     INT,
  -- audit
  source_file     STRING    NOT NULL,
  ingested_at     TIMESTAMP NOT NULL,
  pipeline_run_id STRING    NOT NULL
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Bronze: Product master from Parquet. Append-only. SCD2 handled in silver.';

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- bronze.store_master
-- Source : store_master.csv
-- Frequency: Static — load once, reload on change
-- Not partitioned — tiny table (500 rows)
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.bronze.store_master (
  store_id        STRING,
  store_name      STRING,
  region          STRING,
  city            STRING,
  store_type      STRING,
  opening_date    STRING,
  -- audit
  source_file     STRING    NOT NULL,
  ingested_at     TIMESTAMP NOT NULL,
  pipeline_run_id STRING    NOT NULL
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Bronze: Store master reference data from CSV. SCD2 handled in silver.';

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- bronze.supplier_master
-- Source : supplier_master.csv
-- Frequency: Static — load once, reload on change
-- Not partitioned — tiny table (200 rows)
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.bronze.supplier_master (
  supplier_id           STRING,
  supplier_name         STRING,
  category              STRING,
  performance_rating    DOUBLE,
  on_time_delivery_pct  DOUBLE,
  -- audit
  source_file           STRING    NOT NULL,
  ingested_at           TIMESTAMP NOT NULL,
  pipeline_run_id       STRING    NOT NULL
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Bronze: Supplier master reference data from CSV. SCD2 handled in silver.';

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- bronze.purchase_orders
-- Source : purchase_orders.csv (nested JSON in delivery_notes column)
-- Frequency: Daily + CDC
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.bronze.purchase_orders (
  po_id                     STRING,
  supplier_id               STRING,
  product_id                STRING,
  order_date                STRING,
  expected_delivery_date    STRING,
  actual_delivery_date      STRING,
  quantity_ordered          INT,
  unit_cost                 DOUBLE,
  status                    STRING,
  quality_rating            DOUBLE,
  delivery_notes            STRING,   -- raw nested JSON string, flattened in silver
  -- audit
  source_file               STRING    NOT NULL,
  ingested_at               TIMESTAMP NOT NULL,
  pipeline_run_id           STRING    NOT NULL
)
USING DELTA
PARTITIONED BY (ingested_date STRING)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Bronze: Purchase orders from CSV with nested JSON delivery_notes. Append-only.';

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- bronze.store_inventory
-- Source : store_inventory_snapshots.jsonl
-- Frequency: Every 15 mins
-- temperature_reading is nested JSON string — kept raw in bronze
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.bronze.store_inventory (
  store_id              STRING,
  product_id            STRING,
  current_quantity      INT,
  last_restocked_date   STRING,
  shelf_location        STRING,
  expiry_date           STRING,
  temperature_reading   STRING,   -- raw nested JSON string, flattened in silver
  -- audit
  source_file           STRING    NOT NULL,
  ingested_at           TIMESTAMP NOT NULL,
  pipeline_run_id       STRING    NOT NULL
)
USING DELTA
PARTITIONED BY (ingested_date STRING)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Bronze: Store inventory snapshots from JSONL every 15 mins. Append-only.';

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- bronze.pos_transactions
-- Source : pos_transactions.jsonl (stream) + pos_transactions_sample.csv (batch)
-- Both land in same table — distinguished by source_file audit column
-- Frequency: Real-time (stream) + Daily (batch)
-- Partition : ingested_date (derived from ingested_at in the ingestion job)
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.bronze.pos_transactions (
  transaction_id    STRING,
  store_id          STRING,
  product_id        STRING,
  customer_id       STRING,
  timestamp         STRING,
  quantity          INT,
  unit_price        DOUBLE,
  total_amount      DOUBLE,
  payment_method    STRING,
  channel           STRING,
  _source           STRING        NOT NULL,
  -- audit
  source_file       STRING        NOT NULL,
  ingested_at       TIMESTAMP     NOT NULL,
  pipeline_run_id   STRING        NOT NULL
)
USING DELTA
PARTITIONED BY (ingested_date STRING)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Bronze: POS transactions from JSONL stream and CSV batch. Append-only.';

ALTER TABLE azure3_team7_project.bronze.pos_transactions ADD CONSTRAINT _source_not_null CHECK (_source IS NOT NULL);

ALTER TABLE azure3_team7_project.bronze.pos_transactions ADD CONSTRAINT _source_valid CHECK (_source IN ('pos_realtime_stream', 'pos_batch_csv'));

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- bronze.warehouse_inventory
-- Source : warehouse_inventory.parquet
-- Frequency: Hourly
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.bronze.warehouse_inventory (
  warehouse_id      STRING,
  product_id        STRING,
  current_stock     BIGINT,
  reserved_stock    BIGINT,
  available_stock   BIGINT,
  reorder_level     BIGINT,
  max_stock         BIGINT,
  last_updated      STRING,
  location_zone     STRING,
  -- audit
  source_file       STRING    NOT NULL,
  ingested_at       TIMESTAMP NOT NULL,
  pipeline_run_id   STRING    NOT NULL
)
USING DELTA
PARTITIONED BY (ingested_date STRING)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Bronze: Warehouse inventory snapshots from Parquet. Append-only.';

## SILVER TABLES
Rules:
- NOT NULL enforced on all key/critical columns
- Proper types (no raw strings for dates/timestamps)
- Nested JSON columns flattened
- Business rule CHECK constraints added
- ZORDER defined (applied by maintenance job, documented here)
- SCD2 columns on all master tables
- Partitioned by event date where applicable

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- silver.customers
-- Key      : customer_id
-- Merge on : customer_id
-- ZORDER   : loyalty_tier, zip_code
-- Note     : preferred_categories parsed from JSON string to ARRAY<STRING>
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.silver.customers (
  customer_id           STRING        NOT NULL,
  age_group             STRING,
  gender                STRING,
  zip_code              STRING,
  loyalty_tier          STRING,
  first_purchase_date   DATE,
  total_spend           DECIMAL(10,2),
  preferred_categories  ARRAY<STRING>,
  -- audit
  ingested_at           TIMESTAMP     NOT NULL,
  processed_at          TIMESTAMP     NOT NULL,
  pipeline_run_id       STRING        NOT NULL,
  source_system         STRING        NOT NULL
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Silver: Cleaned customer data. preferred_categories parsed to array.';

ALTER TABLE azure3_team7_project.silver.customers
ADD CONSTRAINT valid_loyalty_tier
CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum'));

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- silver.product_master  — SCD Type 2
-- Key      : product_id + valid_from (one row per version)
-- Merge on : product_id + scd_hash (new hash = new version)
-- ZORDER   : category, supplier_id
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.silver.product_master (
  product_id      STRING        NOT NULL,
  product_name    STRING,
  category        STRING,
  subcategory     STRING,
  brand           STRING,
  supplier_id     STRING,
  cost_price      DECIMAL(10,2),
  selling_price   DECIMAL(10,2),
  weight          DOUBLE,
  length          DOUBLE,
  width           DOUBLE,
  height          DOUBLE,
  status          STRING,
  effective_date  DATE,
  expiry_date     DATE,
  -- SCD2
  scd_hash        STRING        NOT NULL,
  valid_from      TIMESTAMP     NOT NULL,
  valid_to        TIMESTAMP,
  is_current      BOOLEAN       NOT NULL,
  -- audit
  ingested_at     TIMESTAMP     NOT NULL,
  processed_at    TIMESTAMP     NOT NULL,
  pipeline_run_id STRING        NOT NULL,
  source_system   STRING        NOT NULL
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Silver: Product master with SCD Type 2 history. is_current=true for latest version.';

ALTER TABLE azure3_team7_project.silver.product_master
ADD CONSTRAINT valid_product_status
CHECK (status IN ('active', 'discontinued'));

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- silver.store_master  — SCD Type 2
-- Key      : store_id + valid_from
-- Merge on : store_id + scd_hash
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.silver.store_master (
  store_id        STRING    NOT NULL,
  store_name      STRING,
  region          STRING,
  city            STRING,
  store_type      STRING,
  opening_date    DATE,
  -- SCD2
  scd_hash        STRING    NOT NULL,
  valid_from      TIMESTAMP NOT NULL,
  valid_to        TIMESTAMP,
  is_current      BOOLEAN   NOT NULL,
  -- audit
  ingested_at     TIMESTAMP NOT NULL,
  processed_at    TIMESTAMP NOT NULL,
  pipeline_run_id STRING    NOT NULL,
  source_system   STRING    NOT NULL
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Silver: Store master with SCD Type 2 history. is_current=true for latest version.';

ALTER TABLE azure3_team7_project.silver.store_master
ADD CONSTRAINT valid_store_type
CHECK (store_type IN ('Warehouse', 'Superstore', 'Express', 'Online'));

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- silver.supplier_master  — SCD Type 2
-- Key      : supplier_id + valid_from
-- Merge on : supplier_id + scd_hash
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.silver.supplier_master (
  supplier_id           STRING        NOT NULL,
  supplier_name         STRING,
  category              STRING,
  performance_rating    DECIMAL(4,2),
  on_time_delivery_pct  DECIMAL(5,2),
  -- SCD2
  scd_hash              STRING        NOT NULL,
  valid_from            TIMESTAMP     NOT NULL,
  valid_to              TIMESTAMP,
  is_current            BOOLEAN       NOT NULL,
  -- audit
  ingested_at           TIMESTAMP     NOT NULL,
  processed_at          TIMESTAMP     NOT NULL,
  pipeline_run_id       STRING        NOT NULL,
  source_system         STRING        NOT NULL
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Silver: Supplier master with SCD Type 2 history. is_current=true for latest version.';

ALTER TABLE azure3_team7_project.silver.supplier_master
ADD CONSTRAINT valid_performance_rating
CHECK (performance_rating BETWEEN 0 AND 5);

ALTER TABLE azure3_team7_project.silver.supplier_master
ADD CONSTRAINT valid_delivery_pct
CHECK (on_time_delivery_pct BETWEEN 0 AND 100);

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- silver.purchase_orders
-- Key      : po_id (CDC — status updates come in daily)
-- Merge on : po_id
-- ZORDER   : supplier_id, product_id
-- Note     : delivery_notes nested JSON flattened into 4 columns
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.silver.purchase_orders (
  po_id                     STRING        NOT NULL,
  supplier_id               STRING        NOT NULL,
  product_id                STRING        NOT NULL,
  order_date                DATE,
  expected_delivery_date    DATE,
  actual_delivery_date      DATE,
  quantity_ordered          INT,
  unit_cost                 DECIMAL(10,2),
  status                    STRING,
  quality_rating            DECIMAL(4,2),
  -- flattened from delivery_notes
  carrier                   STRING,
  tracking_number           STRING,
  delivery_status           STRING,
  delivery_notes_text       STRING,
  -- audit
  ingested_at               TIMESTAMP     NOT NULL,
  processed_at              TIMESTAMP     NOT NULL,
  pipeline_run_id           STRING        NOT NULL,
  source_system             STRING        NOT NULL
)
USING DELTA
PARTITIONED BY (order_year_month STRING)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Silver: Purchase orders with flattened delivery_notes. Merged on po_id for CDC.';

ALTER TABLE azure3_team7_project.silver.purchase_orders
ADD CONSTRAINT valid_po_status
CHECK (status IN ('pending', 'shipped', 'delivered', 'cancelled'));

ALTER TABLE azure3_team7_project.silver.purchase_orders
ADD CONSTRAINT positive_quantity_ordered
CHECK (quantity_ordered > 0);

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- silver.store_inventory
-- Key      : store_id + product_id (latest snapshot per pair)
-- Merge on : store_id, product_id
-- ZORDER   : store_id, product_id
-- Note     : temperature_reading nested JSON flattened into 3 columns
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.silver.store_inventory (
  store_id              STRING    NOT NULL,
  product_id            STRING    NOT NULL,
  current_quantity      INT       NOT NULL,
  last_restocked_date   DATE,
  shelf_location        STRING,
  expiry_date           DATE,
  -- flattened from temperature_reading
  sensor_id             STRING,
  temperature_celsius   DOUBLE,
  humidity              DOUBLE,
  -- audit
  ingested_at           TIMESTAMP NOT NULL,
  processed_at          TIMESTAMP NOT NULL,
  pipeline_run_id       STRING    NOT NULL,
  source_system         STRING    NOT NULL
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Silver: Latest store inventory per store+product. temperature_reading flattened.';

ALTER TABLE azure3_team7_project.silver.store_inventory
ADD CONSTRAINT non_negative_quantity
CHECK (current_quantity >= 0);

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- silver.pos_transactions
-- Key      : transaction_id (deduplicated here — stream + batch may overlap)
-- Merge on : transaction_id
-- ZORDER   : store_id, product_id
-- Note     : timestamp renamed to transaction_ts to avoid Spark reserved word
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.silver.pos_transactions (
  transaction_id    STRING        NOT NULL,
  store_id          STRING        NOT NULL,
  product_id        STRING        NOT NULL,
  customer_id       STRING,
  transaction_ts    TIMESTAMP     NOT NULL,
  quantity          INT           NOT NULL,
  unit_price        DECIMAL(10,2) NOT NULL,
  total_amount      DECIMAL(10,2) NOT NULL,
  payment_method    STRING,
  channel           STRING,
  -- audit
  ingested_at       TIMESTAMP     NOT NULL,
  processed_at      TIMESTAMP     NOT NULL,
  pipeline_run_id   STRING        NOT NULL,
  source_system     STRING        NOT NULL
)
USING DELTA
PARTITIONED BY (transaction_date STRING)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Silver: Cleaned, deduplicated POS transactions. Merged on transaction_id.';

ALTER TABLE azure3_team7_project.silver.pos_transactions
ADD CONSTRAINT valid_channel
CHECK (channel IN ('online', 'offline'));

ALTER TABLE azure3_team7_project.silver.pos_transactions
ADD CONSTRAINT positive_amount
CHECK (total_amount >= 0);

ALTER TABLE azure3_team7_project.silver.pos_transactions
ADD CONSTRAINT positive_quantity
CHECK (quantity > 0);

In [0]:
%sql
-- -----------------------------------------------------------------------------
-- silver.warehouse_inventory
-- Key      : warehouse_id + product_id (latest snapshot per pair)
-- Merge on : warehouse_id, product_id
-- ZORDER   : product_id, warehouse_id
-- Note     : available_stock recomputed here, not trusted from source
-- -----------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS azure3_team7_project.silver.warehouse_inventory (
  warehouse_id      STRING    NOT NULL,
  product_id        STRING    NOT NULL,
  current_stock     BIGINT    NOT NULL,
  reserved_stock    BIGINT    NOT NULL,
  available_stock   BIGINT    NOT NULL,
  reorder_level     BIGINT,
  max_stock         BIGINT,
  last_updated      TIMESTAMP,
  location_zone     STRING,
  -- audit
  ingested_at       TIMESTAMP NOT NULL,
  processed_at      TIMESTAMP NOT NULL,
  pipeline_run_id   STRING    NOT NULL,
  source_system     STRING    NOT NULL
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableChangeDataFeed'       = 'true'
)
COMMENT 'Silver: Latest warehouse inventory per warehouse+product. Merged on warehouse_id+product_id.';

ALTER TABLE azure3_team7_project.silver.warehouse_inventory
ADD CONSTRAINT non_negative_current_stock
CHECK (current_stock >= 0);

ALTER TABLE azure3_team7_project.silver.warehouse_inventory
ADD CONSTRAINT non_negative_reserved_stock
CHECK (reserved_stock >= 0);